
# ZynNova：MCS 风格 + MCR 风格极复杂异构电极 → 连续隐式曲面 → 非结构化 TetGen → COMSOL

这个 notebook 专门解决两个问题：

1. **MCS/MCR 生成的是体素/统计场，但最终 FEM 网格不能仍然像立方体切出来。**
2. **自由曲面 COMSOL 网格没有 `xmin/xmax` 时，不能错误创建 `x_terminal_pair`。**

最终路线：

```text
MCS-style particle/CBD synthesis
        │
        ├── mixed sphere / ellipsoid particles
        ├── electrostatic packing
        ├── overlap / PSD / CBD controls
        └── individual particle tracking
        │
        ▼
MCR-style characterization
        │
        ├── VolumeFractions
        ├── Variation
        └── Correlations
        │
        ▼
MCR reconstruction → continuous 3-phase probability field
        │
        ├── active probability → particle surface modulation
        └── CBD probability → smooth CBD connected components
        │
        ▼
continuous implicit geometry sampling
        │
        ├── free-form rounded outer electrode
        ├── connected active-material union surface
        └── multiple smooth CBD bodies
        │
        ▼
Marching Cubes only extracts triangular geometry
        │
        │   (the sampling grid is NOT the FEM mesh)
        ▼
one conforming multi-material triangular PLC
        │
        ▼
TetGen 1.6 C++ constrained/conforming Delaunay
        │
        ▼
spatially heterogeneous unstructured Tet4
        │
        ▼
COMSOL MPHTXT / VTK / Gmsh / Abaqus
```

最终只有三个材料域：

```text
1 = active_material
2 = electrolyte
3 = cbd
```

即使 active material 包含多个互相连接或分离的颗粒 cluster，所有 active Tet4 仍共享一个材料 region。


In [ ]:

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import (
    binary_closing,
    binary_opening,
    distance_transform_edt,
    gaussian_filter,
    label as ndi_label,
    zoom,
)

try:
    from skimage.measure import marching_cubes
except ImportError as exc:
    raise RuntimeError(
        "本 notebook 需要 scikit-image 来把连续隐式场提取为三角曲面。"
        "请执行: python -m pip install scikit-image"
    ) from exc

try:
    import zynnova
except ModuleNotFoundError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    explicit = os.environ.get("ZYNNOVA_PROJECT_ROOT")
    if explicit:
        candidates = [Path(explicit), *candidates]
    root = next((p for p in candidates if (p / "src" / "zynnova").is_dir()), None)
    if root is None:
        raise
    sys.path.insert(0, str(root / "src"))
    import zynnova

from zynnova.geometry import (
    TriangleMesh,
    tetrahedron_mean_ratio,
    tetrahedron_signed_volumes,
)

from zynnova.zynmorph import (
    CBDSettings,
    CharacterizationSettings,
    ElectrodeSynthesisConfig,
    FreeformRegion,
    IrregularMeshPolicy,
    LocalRefinementZone,
    PackingSettings,
    ParticleDistribution,
    ReconstructionSettings,
    SurfaceShell,
    assemble_freeform_plc,
    audit_freeform_shell_clearance,
    audit_multiphase_plc,
    characterize,
    export_fem_mesh,
    generate_particle_electrode,
    inspect_comsol_mphtxt,
    load_comsol_tet4_mphtxt,
    mesh_complex_regions,
    orient_closed_surface,
    reconstruct,
    regular_tetrahedron_volume_from_edge,
    tetgen_native_diagnostics,
    tetgen_native_status,
)

print("ZynNova:", zynnova.__file__)
print("Python :", sys.executable)


## 1. 复杂度参数

In [ ]:

SEED = 20260818

# MCS 风格源微结构
SOURCE_SHAPE_ZYX = (32, 32, 32)
VOXEL_SIZE_M = 0.80e-6
ACTIVE_TARGET = 0.56

# 隐式连续几何采样分辨率。
# 注意：这个网格只用于 scalar field sampling / marching cubes，
# 不是最终 FEM 网格。
IMPLICIT_GRID_N = 48

# 自由形状电极外壳半轴
OUTER_AXES_M = np.array([18.0, 16.0, 14.0]) * 1e-6
OUTER_SUPERELLIPSOID_P = 6.0

ACTIVE = 1
ELECTROLYTE = 2
CBD = 3

REGION_NAMES = {
    ACTIVE: "active_material",
    ELECTROLYTE: "electrolyte",
    CBD: "cbd",
}

# 在用户 Windows 环境默认真正运行 native TetGen。
RUN_TETGEN = os.environ.get("ZYNNOVA_RUN_NATIVE_TETGEN", "1") == "1"
MAXIMUM_TETRAHEDRA = 6_000_000

OUTPUT_DIR = Path("zynnova_runs") / "mcs_mcr_heterogeneous_implicit_electrode"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_TETGEN:", RUN_TETGEN)
print("Output:", OUTPUT_DIR)



## 2. MCS 风格生成极复杂体素电极

这里使用 ZynNova 中已经统一的 MCS-CICE 风格能力：

- `geometry="mixed"`：球 + 椭球；
- 三轴方向与尺寸随机；
- `electrostatic` packing；
- 允许少量粒子接触/重叠；
- mixed CBD；
- CBD nanoporosity；
- 每颗粒保留 tracking ID；
- 最终 active/CBD/electrolyte 三相严格统计。


In [ ]:

mcs_config = ElectrodeSynthesisConfig(
    shape_zyx=SOURCE_SHAPE_ZYX,
    voxel_size_m=VOXEL_SIZE_M,
    active_volume_fraction=ACTIVE_TARGET,
    active_phase=ACTIVE,
    electrolyte_phase=ELECTROLYTE,
    cbd_phase=CBD,
    seed=SEED,

    particle_distribution=ParticleDistribution(
        geometry="mixed",
        particle_count=40,
        median_diameter_vox=7.6,
        lognormal_sigma=0.30,
        minimum_diameter_vox=4.0,
        maximum_diameter_vox=12.0,
        sphere_fraction=0.22,
        axis_ratio_ranges=((0.55, 1.45), (0.50, 1.55)),
    ),

    packing=PackingSettings(
        method="electrostatic",
        boundary_mode="contained",
        overlap_fraction=0.08,
        max_attempts_per_particle=1200,
    ),

    cbd=CBDSettings(
        method="mixed",
        target_volume_fraction=0.11,
        nanoporosity=0.35,
        correlation_length_vox=2.2,
        bridge_radius_vox=1.3,
        interface_decay_vox=2.8,
        mixed_bridge_fraction=0.55,
    ),

    individual_particle_labels=True,
    particle_label_offset=1000,
)

mcs = generate_particle_electrode(mcs_config)

print("Accepted particles:", len(mcs.particles))
print("PSD validation:", mcs.psd_validation)
print("Statistics:", mcs.statistics)

phase_ids, phase_counts = np.unique(mcs.volume.labels, return_counts=True)
display(pd.DataFrame({
    "phase": phase_ids,
    "count": phase_counts,
    "fraction": phase_counts / phase_counts.sum(),
}))

particle_table = pd.DataFrame([
    {
        "particle_id": p.particle_id,
        "geometry": p.geometry,
        "diameter_vox": p.nominal_diameter_vox,
        "radius_z_vox": p.radii_zyx_vox[0],
        "radius_y_vox": p.radii_zyx_vox[1],
        "radius_x_vox": p.radii_zyx_vox[2],
        "inserted_voxels": p.inserted_voxels,
        "overlap_fraction": p.overlap_fraction,
    }
    for p in mcs.particles
])

display(particle_table)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

mid = SOURCE_SHAPE_ZYX[0] // 2

axes[0].imshow(mcs.volume.labels[mid], origin="lower", cmap="tab10")
axes[0].set_title("MCS source phases")

axes[1].imshow(mcs.particle_labels[mid], origin="lower", cmap="nipy_spectral")
axes[1].set_title("individual particle tracking IDs")

axes[2].hist(
    [p.nominal_diameter_vox for p in mcs.particles],
    bins=14,
)
axes[2].set_title("particle-size distribution")
axes[2].set_xlabel("nominal diameter [voxel]")

plt.tight_layout()
plt.show()



## 3. MCR 风格 characterization → reconstruction

MCR 部分不是只读取一个体积分数。

这里用：

```text
VolumeFractions
Variation
Correlations
```

一起约束三相空间统计。

重构输出的 `probabilities` 是严格 softmax 多相概率场：

\[
p_\mathrm{elyte}+p_\mathrm{active}+p_\mathrm{CBD}=1
\]

后面直接利用这个连续概率场创建复杂的连续几何，而不是把 `argmax` voxel 当成最终 FEM 几何。


In [ ]:

characterization_settings = CharacterizationSettings(
    descriptor_types=(
        "VolumeFractions",
        "Variation",
        "Correlations",
    ),
    limit_to=6,
    use_multiphase=True,
    use_multigrid_descriptors=False,
    periodic=True,
    slice_mode="full",
    isotropic=False,
    phase_ids=(ELECTROLYTE, ACTIVE, CBD),
)

target = characterize(
    mcs.volume.labels,
    characterization_settings,
)

print("Descriptors:", target.descriptor_types)
print("Target volume fractions:", target.metadata["volume_fractions"])

reconstruction_settings = ReconstructionSettings(
    descriptor_types=target.descriptor_types,
    descriptor_weights=(200.0, 1.0, 4.0),
    optimizer_type="Adam",
    loss_type="MSE",
    limit_to=6,
    use_multiphase=True,
    use_multigrid_descriptors=False,
    use_multigrid_reconstruction=False,
    periodic=True,
    slice_mode="full",
    isotropic=False,
    learning_rate=0.05,
    max_iter=50,
    tolerance=1.0e-5,
    seed=SEED + 1,
    device="auto",
    dtype="float32",
    phase_ids=(ELECTROLYTE, ACTIVE, CBD),
)

mcr = reconstruct(
    target,
    desired_shape=(18, 18, 18),
    settings=reconstruction_settings,
)

assert mcr.probabilities is not None
assert np.allclose(
    mcr.probabilities.sum(axis=0),
    1.0,
    atol=2.0e-6,
)

print("Final MCR loss:", mcr.final_loss)
print(
    "Reconstructed mean probabilities:",
    mcr.probabilities.mean(axis=(1, 2, 3)),
)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))

z = mcr.probabilities.shape[1] // 2

for ax, phase_index, title in zip(
    axes,
    range(3),
    ("electrolyte probability", "active probability", "CBD probability"),
    strict=True,
):
    im = ax.imshow(
        mcr.probabilities[phase_index, z],
        origin="lower",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.75)

plt.tight_layout()
plt.show()



## 4. 把 MCS + MCR 转成连续隐式几何

关键原则：

> 体素只负责产生粒子统计、packing 与 descriptor target；最终几何通过连续 scalar field 重建。

### Active material

每一个 MCS `ParticleRecord` 被恢复成旋转的连续 superellipsoid。  
允许 MCS 中相互接触/轻微重叠的粒子通过 `min()` 做**几何 union**，因此不会产生互穿的 PLC shell。

### CBD

MCR 的 CBD 概率场经过三次插值 + Gaussian regularization，选择高概率区域形成多个平滑 CBD connected components。

### Outer body

外壳使用高阶 superellipsoid，不是 box。


In [ ]:

N = IMPLICIT_GRID_N

mins = -1.08 * OUTER_AXES_M
maxs = +1.08 * OUTER_AXES_M

x = np.linspace(mins[0], maxs[0], N)
y = np.linspace(mins[1], maxs[1], N)
z = np.linspace(mins[2], maxs[2], N)

zz, yy, xx = np.meshgrid(z, y, x, indexing="ij")

spacing_zyx = (
    z[1] - z[0],
    y[1] - y[0],
    x[1] - x[0],
)

# 非矩形外部电极体。
outer_measure = (
    (np.abs(xx) / OUTER_AXES_M[0]) ** OUTER_SUPERELLIPSOID_P
    + (np.abs(yy) / OUTER_AXES_M[1]) ** OUTER_SUPERELLIPSOID_P
    + (np.abs(zz) / OUTER_AXES_M[2]) ** OUTER_SUPERELLIPSOID_P
)

# 连续几何使用更严格的内层安全壳。后续 active/CBD 的零等值面
# 都会显式裁剪到这个自由形状内层，而不是只裁 mask。
ACTIVE_OUTER_MEASURE_LIMIT = 0.78
CBD_OUTER_MEASURE_LIMIT = 0.60
CBD_ACTIVE_CLEARANCE_M = 1.80e-6

outer_safe = outer_measure <= ACTIVE_OUTER_MEASURE_LIMIT


In [ ]:

# MCS particle records -> continuous active-material union level set.

shape_zyx = np.asarray(SOURCE_SHAPE_ZYX, dtype=float)
source_center_zyx = (shape_zyx - 1.0) / 2.0

# zyx <-> xyz permutation
P = np.array(
    [
        [0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
        [1.0, 0.0, 0.0],
    ]
)

active_field = np.full((N, N, N), np.inf, dtype=np.float32)
shape_rng = np.random.default_rng(SEED + 2)

for record in mcs.particles:
    center_xyz = (
        np.asarray(record.center_zyx_vox, dtype=float) - source_center_zyx
    )[::-1] * VOXEL_SIZE_M

    radii_xyz = (
        np.asarray(record.radii_zyx_vox, dtype=float)[::-1]
        * VOXEL_SIZE_M
        * 0.94
    )

    rotation_zyx = np.asarray(record.rotation_matrix, dtype=float)
    rotation_xyz = P @ rotation_zyx @ P.T

    dx = xx - center_xyz[0]
    dy = yy - center_xyz[1]
    dz = zz - center_xyz[2]

    # row-vector local = delta @ R
    l0 = dx * rotation_xyz[0, 0] + dy * rotation_xyz[1, 0] + dz * rotation_xyz[2, 0]
    l1 = dx * rotation_xyz[0, 1] + dy * rotation_xyz[1, 1] + dz * rotation_xyz[2, 1]
    l2 = dx * rotation_xyz[0, 2] + dy * rotation_xyz[1, 2] + dz * rotation_xyz[2, 2]

    exponent = (
        2.0
        if record.geometry == "sphere"
        else float(shape_rng.uniform(1.6, 3.5))
    )

    distance = (
        np.abs(l0 / radii_xyz[0]) ** exponent
        + np.abs(l1 / radii_xyz[1]) ** exponent
        + np.abs(l2 / radii_xyz[2]) ** exponent
    ) ** (1.0 / exponent)

    active_field = np.minimum(
        active_field,
        distance.astype(np.float32),
    )

# 用 MCR active probability 给联合表面施加低频统计扰动。
mcr_active = zoom(
    mcr.probabilities[1],
    np.asarray((N, N, N)) / np.asarray(mcr.probabilities[1].shape),
    order=3,
    mode="wrap",
)[:N, :N, :N]

active_modulation = gaussian_filter(
    mcr_active - mcr_active.mean(),
    sigma=2.0,
)
active_modulation /= max(float(np.std(active_modulation)), 1.0e-8)

active_field = active_field / (
    1.0 + 0.04 * np.tanh(active_modulation)
)

# 几何交集必须在 level-set 本身完成。上一版只对 active_mask 加了
# outer_safe，但 Marching Cubes 仍使用未裁剪 active_field，可能让 active
# surface 穿出 outer shell。这里用 implicit intersection：
# max(phi_active, phi_inner_outer) <= 0。
active_geometry_field = np.maximum(
    active_field - 1.0,
    outer_measure / ACTIVE_OUTER_MEASURE_LIMIT - 1.0,
).astype(np.float32)

active_mask = active_geometry_field <= 0.0

print("Continuous active sampling fraction:", float(active_mask.mean()))


In [ ]:

# Marching Cubes 提取 active union。
# 注意：这里输出的是任意方向三角曲面，不是 FEM 六面体。

active_vertices_zyx, active_faces, _, _ = marching_cubes(
    active_geometry_field,
    level=0.0,
    spacing=spacing_zyx,
)

active_vertices_xyz = (
    active_vertices_zyx[:, ::-1]
    + np.array([x[0], y[0], z[0]])
)

active_surface = orient_closed_surface(
    TriangleMesh(
        vertices=active_vertices_xyz,
        faces=active_faces,
        metadata={
            "source": "MCS particle union + MCR active modulation",
            "voxel_faces_used": False,
        },
    )
)

print(
    "Active surface:",
    active_surface.n_vertices,
    "vertices /",
    active_surface.n_faces,
    "triangles",
)


## 5. 从 MCR CBD probability 生成多个平滑 CBD 连通体

In [ ]:

mcr_cbd = zoom(
    mcr.probabilities[2],
    np.asarray((N, N, N)) / np.asarray(mcr.probabilities[2].shape),
    order=3,
    mode="wrap",
)[:N, :N, :N]

cbd_score = gaussian_filter(mcr_cbd, sigma=1.0)

# 以物理距离而不是 dimensionless particle-field 阈值控制 active/CBD
# 间隙；同时给自由外壳留出独立的连续几何安全带。
distance_from_active_m = distance_transform_edt(
    ~active_mask,
    sampling=spacing_zyx,
)

cbd_allowed = (
    (outer_measure <= CBD_OUTER_MEASURE_LIMIT)
    & (distance_from_active_m >= CBD_ACTIVE_CLEARANCE_M)
)

if not np.any(cbd_allowed):
    raise RuntimeError("CBD clearance constraints removed the entire admissible region")

threshold = float(np.quantile(cbd_score[cbd_allowed], 0.80))
cbd_mask = (cbd_score >= threshold) & cbd_allowed

cbd_mask = binary_opening(cbd_mask, iterations=1)
cbd_mask = binary_closing(cbd_mask, iterations=1)
# Morphology can regrow a voxel into the forbidden band; enforce the clearance
# again before continuous-surface extraction.
cbd_mask &= cbd_allowed

component_labels, component_count = ndi_label(cbd_mask)
component_sizes = np.bincount(component_labels.ravel())

# 只保留最大的多个复杂 connected components，去掉像素级小噪声。
component_ids = np.argsort(component_sizes[1:])[::-1] + 1

cbd_filtered = np.zeros_like(cbd_mask)
kept_components = []

for component_id in component_ids[:14]:
    size = int(component_sizes[component_id])
    if size < 12:
        continue
    cbd_filtered |= component_labels == component_id
    kept_components.append((int(component_id), size))

print("MCR-derived CBD components:", len(kept_components))
print("Largest component sizes:", kept_components[:8])


In [ ]:

# 连续化 CBD mask 后再提取表面。
cbd_continuous = gaussian_filter(
    cbd_filtered.astype(np.float32),
    sigma=0.72,
)
# Gaussian smoothing may expand the 0.5 isosurface by a fraction of a sampling
# cell. Hard-mask the physical forbidden band once more; the original band is
# deliberately several cells wide.
cbd_continuous[~cbd_allowed] = 0.0

cbd_vertices_zyx, cbd_faces, _, _ = marching_cubes(
    cbd_continuous,
    level=0.50,
    spacing=spacing_zyx,
)

cbd_vertices_xyz = (
    cbd_vertices_zyx[:, ::-1]
    + np.array([x[0], y[0], z[0]])
)

cbd_surface = orient_closed_surface(
    TriangleMesh(
        vertices=cbd_vertices_xyz,
        faces=cbd_faces,
        metadata={
            "source": "MCR reconstructed CBD probability",
            "voxel_faces_used": False,
        },
    )
)

print(
    "CBD surface:",
    cbd_surface.n_vertices,
    "vertices /",
    cbd_surface.n_faces,
    "triangles",
)


## 6. 自由外壳 + 多域 region seeds

In [ ]:

# Outer shell 也由连续 implicit function 提取，宏观几何不是 cuboid。
outer_vertices_zyx, outer_faces, _, _ = marching_cubes(
    outer_measure.astype(np.float32),
    level=1.0,
    spacing=spacing_zyx,
)

outer_vertices_xyz = (
    outer_vertices_zyx[:, ::-1]
    + np.array([x[0], y[0], z[0]])
)

outer_surface = orient_closed_surface(
    TriangleMesh(
        vertices=outer_vertices_xyz,
        faces=outer_faces,
        metadata={
            "source": "free-form superellipsoid electrode body",
            "rectangular": False,
        },
    )
)

print(
    "Outer surface:",
    outer_surface.n_vertices,
    "vertices /",
    outer_surface.n_faces,
    "triangles",
)


In [ ]:

# 对 active / CBD 每个 disconnected component 自动生成 interior region seed。

connectivity = np.ones((3, 3, 3), dtype=int)

active_components, n_active_components = ndi_label(
    active_mask,
    structure=connectivity,
)
active_distance = distance_transform_edt(active_mask)

active_seeds = []

for component in range(1, n_active_components + 1):
    loc = np.where(active_components == component)
    if len(loc[0]) < 12:
        continue
    score = active_distance[loc]
    k = int(np.argmax(score))
    iz, iy, ix = (int(axis[k]) for axis in loc)
    active_seeds.append(
        (float(x[ix]), float(y[iy]), float(z[iz]))
    )

cbd_components, n_cbd_components = ndi_label(
    cbd_filtered,
    structure=connectivity,
)
cbd_distance = distance_transform_edt(cbd_filtered)

cbd_seeds = []

for component in range(1, n_cbd_components + 1):
    loc = np.where(cbd_components == component)
    if len(loc[0]) < 12:
        continue
    score = cbd_distance[loc]
    k = int(np.argmax(score))
    iz, iy, ix = (int(axis[k]) for axis in loc)
    cbd_seeds.append(
        (float(x[ix]), float(y[iy]), float(z[iz]))
    )

# 外层电解液 seed 放在靠近自由外壳、但远离内部相的位置。
electrolyte_seed = (
    float(-0.85 * OUTER_AXES_M[0]),
    0.0,
    0.0,
)

print("Active connected components:", len(active_seeds))
print("CBD connected components   :", len(cbd_seeds))
print("Electrolyte seed           :", electrolyte_seed)


## 7. 组装一个共形 multi-material PLC

In [ ]:

shells = (
    SurfaceShell(
        surface=outer_surface,
        inside_region=ELECTROLYTE,
        outside_region=None,
        name="freeform_outer_electrode",
    ),
    SurfaceShell(
        surface=active_surface,
        inside_region=ACTIVE,
        outside_region=ELECTROLYTE,
        name="active_material_union",
    ),
    SurfaceShell(
        surface=cbd_surface,
        inside_region=CBD,
        outside_region=ELECTROLYTE,
        name="mcr_cbd_network",
    ),
)

regions = (
    FreeformRegion(
        region=ELECTROLYTE,
        seed_m_xyz=electrolyte_seed,
        name="electrolyte",
    ),
    *(
        FreeformRegion(
            region=ACTIVE,
            seed_m_xyz=seed,
            name=f"active_component_{index:02d}",
        )
        for index, seed in enumerate(active_seeds)
    ),
    *(
        FreeformRegion(
            region=CBD,
            seed_m_xyz=seed,
            name=f"cbd_component_{index:02d}",
        )
        for index, seed in enumerate(cbd_seeds)
    ),
)

clearance_audit = audit_freeform_shell_clearance(
    shells,
    minimum_clearance_factor=0.40,
    minimum_clearance_m=0.35e-6,
)

print("Free-form shell clearance audit:")
for item in clearance_audit.pair_results:
    print(
        f"  {item.left_name} <-> {item.right_name}: "
        f"gap={item.minimum_sample_distance_m * 1e6:.3f} um, "
        f"reference_edge={item.reference_edge_length_m * 1e6:.3f} um, "
        f"ratio={item.clearance_ratio:.3f}, valid={item.valid}"
    )

assert clearance_audit.valid, (
    "The implicit surfaces are too close for robust TetGen facet recovery. "
    "Increase the physical material clearance before tetrahedralization."
)

plc = assemble_freeform_plc(
    shells,
    strict=True,
    minimum_clearance_factor=0.40,
    minimum_clearance_m=0.35e-6,
    strict_clearance=True,
)

plc_audit = audit_multiphase_plc(plc)

print(plc_audit)
print("PLC vertices :", len(plc.vertices))
print("PLC triangles:", len(plc.triangles))
print("PLC regions  :", plc.regions)

assert plc_audit.valid
assert set(plc.regions) == {ACTIVE, ELECTROLYTE, CBD}


In [ ]:

# 可视化 PLC 三角面重心，确认没有立方体表面台阶。
count = min(18000, len(plc.triangles))
ids = np.linspace(0, len(plc.triangles) - 1, count, dtype=int)

tri_xyz = plc.vertices[plc.triangles[ids]]
centers = tri_xyz.mean(axis=1) * 1e6

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    centers[:, 0],
    centers[:, 1],
    centers[:, 2],
    s=1,
    alpha=0.32,
)
ax.set_xlabel("x [um]")
ax.set_ylabel("y [um]")
ax.set_zlabel("z [um]")
ax.set_title("Continuous heterogeneous PLC — not voxel cube faces")
plt.show()


## 8. TetGen native 检查

In [ ]:

status = tetgen_native_status()

print(json.dumps({
    "available": status.available,
    "version": status.version,
    "license": status.license,
    "reason": status.reason,
    "module_path": None if status.module_path is None else str(status.module_path),
}, indent=2, ensure_ascii=False))

if not status.available:
    print(
        json.dumps(
            tetgen_native_diagnostics(),
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

if RUN_TETGEN and not status.available:
    raise RuntimeError(
        "TetGen native backend unavailable. "
        '执行 python -m pip install -e ".[zynmorph-all]" -v，'
        "重启 Jupyter kernel，然后运行 scripts/diagnose_tetgen_native.py。"
    )


## 9. 真正的异构空间 sizing

In [ ]:

# active/CBD interface 比 bulk electrolyte 更细；
# 再在部分 active / CBD component 附近加入独立局部细化。
local_zones = []

for index, seed in enumerate(active_seeds[:3]):
    local_zones.append(
        LocalRefinementZone(
            center_m_xyz=seed,
            radius_m=4.0e-6,
            maximum_tetra_volume_m3=regular_tetrahedron_volume_from_edge(0.72e-6),
            name=f"active_hotspot_{index}",
        )
    )

for index, seed in enumerate(cbd_seeds[:4]):
    local_zones.append(
        LocalRefinementZone(
            center_m_xyz=seed,
            radius_m=3.0e-6,
            maximum_tetra_volume_m3=regular_tetrahedron_volume_from_edge(0.58e-6),
            name=f"cbd_hotspot_{index}",
        )
    )

policy = IrregularMeshPolicy(
    base_edge_length_m=2.50e-6,

    region_edge_lengths_m={
        ACTIVE: 1.40e-6,
        ELECTROLYTE: 2.20e-6,
        CBD: 0.95e-6,
    },

    interface_edge_lengths_m={
        (ACTIVE, ELECTROLYTE): 0.78e-6,
        (CBD, ELECTROLYTE): 0.60e-6,
    },

    local_refinement_zones=tuple(local_zones),

    radius_edge_ratio=1.42,
    minimum_dihedral_degrees=8.0,
    optimization_level=2,

    # 已经是连续自由曲面，不需要 voxel-junction repair。
    regularize_junctions=False,
    smoothing_iterations=0,

    normalize_coordinates=True,
    consistency_check=True,
    conforming_delaunay=True,

    # Free-form TetGen facet-recovery protection.  The first pass keeps full
    # conforming Delaunay refinement; a split_subface/very-close-facet failure
    # retries with TetGen -Y, no boundary area splitting, and volume/local sizing.
    freeform_minimum_clearance_factor=0.40,
    freeform_minimum_clearance_m=0.35e-6,
    freeform_strict_clearance=True,
    preserve_boundary_facets=False,
    retry_preserve_boundary_on_facet_error=True,

    quiet=False,
)

print(policy)


## 10. TetGen C++ 非结构化四面体化

In [ ]:

fem = None

if RUN_TETGEN:
    fem = mesh_complex_regions(
        shells,
        regions=regions,
        policy=policy,
        maximum_tetrahedra=MAXIMUM_TETRAHEDRA,
    )

    print("Backend:", fem.backend)
    print("Nodes  :", fem.mesh.n_nodes)
    print("Tet4   :", fem.mesh.n_cells)
    print("Regions:", sorted(map(int, np.unique(fem.mesh.cell_regions))))
    print("Quality:", fem.quality)

    assert fem.quality.fem_ready
    assert fem.quality.inverted_cells == 0
    assert fem.quality.degenerate_cells == 0
    assert set(map(int, np.unique(fem.mesh.cell_regions))) == {
        ACTIVE,
        ELECTROLYTE,
        CBD,
    }
else:
    print(
        "TetGen skipped only because ZYNNOVA_RUN_NATIVE_TETGEN=0. "
        "All MCS/MCR/implicit-surface/PLC steps above were executed."
    )


## 11. 硬性证明最终网格不是 cube-derived Tet4

In [ ]:

if fem is not None:
    mesh = fem.mesh

    volumes = np.abs(tetrahedron_signed_volumes(mesh))
    mean_ratio = tetrahedron_mean_ratio(mesh)

    pairs = (
        (0, 1), (0, 2), (0, 3),
        (1, 2), (1, 3), (2, 3),
    )

    edge_vectors = np.concatenate(
        [
            mesh.nodes[mesh.tetrahedra[:, j]]
            - mesh.nodes[mesh.tetrahedra[:, i]]
            for i, j in pairs
        ],
        axis=0,
    )

    edge_lengths = np.linalg.norm(edge_vectors, axis=1)
    unit = edge_vectors / np.maximum(edge_lengths[:, None], 1.0e-30)

    axis_aligned_fraction = float(
        np.mean(
            np.max(np.abs(unit), axis=1) > 0.9995
        )
    )

    vq = np.quantile(volumes, [0.01, 0.05, 0.50, 0.95, 0.99])
    eq = np.quantile(edge_lengths, [0.01, 0.05, 0.50, 0.95, 0.99])

    print("Tet volume q01/q05/q50/q95/q99 [um^3]:", vq * 1e18)
    print("Edge q01/q05/q50/q95/q99 [um]:", eq * 1e6)
    print("q95/q05 volume:", float(vq[3] / vq[1]))
    print("q95/q05 edge  :", float(eq[3] / eq[1]))
    print("CV(volume)    :", float(np.std(volumes) / np.mean(volumes)))
    print("Axis-aligned edge fraction:", axis_aligned_fraction)
    print("Median mean-ratio:", float(np.median(mean_ratio)))

    assert vq[3] / vq[1] > 1.35
    assert eq[3] / eq[1] > 1.20
    assert np.std(volumes) / np.mean(volumes) > 0.08
    assert axis_aligned_fraction < 0.35


In [ ]:

if fem is not None:
    centers = fem.mesh.nodes[fem.mesh.tetrahedra].mean(axis=1)

    # 取 y≈0 截面附近的 tetra centroids。
    band = np.abs(centers[:, 1]) < 0.70e-6
    section = centers[band] * 1e6
    section_regions = fem.mesh.cell_regions[band]

    if len(section) > 25000:
        ids = np.linspace(0, len(section) - 1, 25000, dtype=int)
        section = section[ids]
        section_regions = section_regions[ids]

    plt.figure(figsize=(10, 7))
    plt.scatter(
        section[:, 0],
        section[:, 2],
        c=section_regions,
        s=3,
        alpha=0.45,
        cmap="tab10",
    )
    plt.xlabel("x [um]")
    plt.ylabel("z [um]")
    plt.title("Unstructured Tet4 centroid slice")
    plt.show()



## 12. COMSOL MPHTXT

这里故意保持：

```python
include_coordinate_boundaries=True
include_default_boundary_unions=True
```

因为这是对本次源码修复的回归测试。

自由曲面没有完整的 `xmin/xmax` 平面时：

```text
x_terminal_pair
y_periodic_candidate_pair
z_periodic_candidate_pair
transverse_boundaries
```

不会再错误地引用不存在的 coordinate sets。

用户自己显式提供错误的 boundary union 时仍然会抛错。


In [ ]:

exports = None
mphtxt_path = None

if fem is not None:
    exports = export_fem_mesh(
        fem,
        OUTPUT_DIR / "fem",
        formats=("mphtxt", "vtk", "msh", "inp"),
        export_boundary=True,

        comsol_domain_selections={
            "active_material": (ACTIVE,),
            "electrolyte": (ELECTROLYTE,),
            "cbd": (CBD,),
            "solid_material": (ACTIVE, CBD),
            "entire_electrode": (ACTIVE, ELECTROLYTE, CBD),
        },

        comsol_options={
            "include_coordinate_boundaries": True,
            "include_default_boundary_unions": True,

            # custom regions，不套用 battery phase enum 的默认组合。
            "include_default_battery_selections": False,

            "include_boundaries": True,
            "include_internal_interfaces": True,
            "include_exterior": True,
            "include_domain_entity_indices": True,
            "verify": True,
        },
    )

    print("Exports:")
    for key, value in exports.exports.items():
        print(f"  {key}: {value}")

    mphtxt_path = exports.exports["mphtxt"]

    info = inspect_comsol_mphtxt(mphtxt_path)
    print("\nMPHTXT:")
    print(info)

    selection_names = {item.label for item in info.selections}

    # 对自由曲面，若没有真实 coordinate face，就不应该创建这些 union。
    for optional_union in (
        "x_terminal_pair",
        "y_periodic_candidate_pair",
        "z_periodic_candidate_pair",
        "transverse_boundaries",
    ):
        print(
            optional_union,
            "present" if optional_union in selection_names else "skipped (correct for free-form)",
        )

    roundtrip = load_comsol_tet4_mphtxt(mphtxt_path)

    print(
        "Round-trip regions:",
        sorted(map(int, np.unique(roundtrip.cell_regions))),
    )

    assert roundtrip.n_nodes == fem.mesh.n_nodes
    assert roundtrip.n_cells == fem.mesh.n_cells
    assert set(map(int, np.unique(roundtrip.cell_regions))) == {
        ACTIVE,
        ELECTROLYTE,
        CBD,
    }

    print("\nCOMSOL round-trip: PASS")


## 13. 最终审计报告

In [ ]:

summary = {
    "schema": "zynnova.mcs-mcr-heterogeneous-implicit-electrode.v1",

    "mcs": {
        "accepted_particles": len(mcs.particles),
        "packing": mcs_config.packing.method,
        "source_active_fraction": mcs.statistics.active_fraction,
        "source_cbd_geometric_fraction": mcs.statistics.cbd_geometric_fraction,
        "source_electrolyte_fraction": mcs.statistics.electrolyte_fraction,
        "psd_ks": mcs.psd_validation.ks_statistic,
    },

    "mcr": {
        "descriptors": list(target.descriptor_types),
        "final_loss": mcr.final_loss,
        "mean_probabilities": mcr.probabilities.mean(axis=(1, 2, 3)).tolist(),
    },

    "continuous_geometry": {
        "implicit_sampling_grid": [N, N, N],
        "sampling_grid_is_fem_mesh": False,
        "outer_rectangular": False,
        "active_surface_vertices": active_surface.n_vertices,
        "active_surface_triangles": active_surface.n_faces,
        "active_connected_components": len(active_seeds),
        "cbd_surface_vertices": cbd_surface.n_vertices,
        "cbd_surface_triangles": cbd_surface.n_faces,
        "cbd_connected_components": len(cbd_seeds),
        "plc_vertices": int(len(plc.vertices)),
        "plc_triangles": int(len(plc.triangles)),
        "plc_valid": bool(plc_audit.valid),
    },

    "tetgen_requested": bool(RUN_TETGEN),
    "tetgen_available": bool(status.available),
}

if fem is not None:
    volumes = np.abs(tetrahedron_signed_volumes(fem.mesh))

    pairs = ((0,1),(0,2),(0,3),(1,2),(1,3),(2,3))
    edge_vectors = np.concatenate([
        fem.mesh.nodes[fem.mesh.tetrahedra[:, j]]
        - fem.mesh.nodes[fem.mesh.tetrahedra[:, i]]
        for i, j in pairs
    ])
    edge_lengths = np.linalg.norm(edge_vectors, axis=1)
    unit = edge_vectors / np.maximum(edge_lengths[:, None], 1.0e-30)

    summary["mesh"] = {
        "backend": fem.backend,
        "nodes": fem.mesh.n_nodes,
        "tetrahedra": fem.mesh.n_cells,
        "regions": sorted(map(int, np.unique(fem.mesh.cell_regions))),
        "inverted": fem.quality.inverted_cells,
        "degenerate": fem.quality.degenerate_cells,
        "median_mean_ratio": fem.quality.median_mean_ratio,
        "volume_q05_um3": float(np.quantile(volumes, 0.05) * 1e18),
        "volume_q50_um3": float(np.quantile(volumes, 0.50) * 1e18),
        "volume_q95_um3": float(np.quantile(volumes, 0.95) * 1e18),
        "edge_q05_um": float(np.quantile(edge_lengths, 0.05) * 1e6),
        "edge_q50_um": float(np.quantile(edge_lengths, 0.50) * 1e6),
        "edge_q95_um": float(np.quantile(edge_lengths, 0.95) * 1e6),
        "axis_aligned_edge_fraction": float(
            np.mean(np.max(np.abs(unit), axis=1) > 0.9995)
        ),
        "mphtxt": str(mphtxt_path),
    }

report_path = OUTPUT_DIR / "validation_summary.json"
report_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nValidation:", report_path)

if fem is not None:
    print("FINAL MPHTXT:", mphtxt_path)
